# 3.2 上下文窗口 / The context window

模型一次能读多少，读不下时会怎样，以及为什么对话越长每一轮越贵。

How much a model can read at once, what happens when it does not fit, and why each turn of a longer conversation costs more.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. token 计费与窗口管理 / Token billing and window management

实现四个小工具，即统计 token 数、计算多轮对话累计计费的输入和输出 token、按单价计算费用，以及在超出窗口时丢弃最早的消息。测试会验证轮数翻倍时累计输入约变为 4 倍。价格以参数传入，不绑定任何一家的实际定价。

Implement four small tools, namely counting tokens, computing the input and output tokens billed over a multi-turn conversation, computing the cost from unit prices, and dropping the oldest messages when the window is exceeded. The tests verify that doubling the number of turns makes the cumulative input about four times as large. Prices are passed as arguments and are not tied to the actual pricing of any provider.

需要联网下载模型。/ Downloads a model.

In [ ]:
import tiktoken


def count_tokens(enc, text):
    """Number of tokens of `text` under the tiktoken encoding `enc`."""
    # TODO 1
    raise NotImplementedError


def chat_billed_tokens(user_lens, assistant_lens, system_len=0):
    """Tokens billed over a whole multi-turn conversation.

    Turn i sends the system prompt, every earlier user and assistant message, and the new
    user message as INPUT, and receives assistant_lens[i] tokens as OUTPUT.
    user_lens and assistant_lens are lists of equal length (token counts per turn).
    Returns (total_input_tokens, total_output_tokens).
    """
    # TODO 2
    raise NotImplementedError


def cost(input_tokens, output_tokens, price_in, price_out):
    """Cost in the currency of the prices, which are given per ONE MILLION tokens."""
    # TODO 3
    raise NotImplementedError


def fit_to_window(message_lens, limit, keep_first=True):
    """Drop the oldest messages until the conversation fits into `limit` tokens.

    message_lens: token counts of the messages, oldest first. If keep_first is True the
    first message (the system prompt) is never dropped.
    Returns the list of indices of the messages that are kept, in order.
    """
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
assert count_tokens(enc, "Hello world") == 2
assert count_tokens(enc, "北京是中国的首都") == 6

# three turns: user 10, 20, 30 tokens; assistant 100 each; system prompt 50
i, o = chat_billed_tokens([10, 20, 30], [100, 100, 100], system_len=50)
assert o == 300
assert i == (50 + 10) + (50 + 10 + 100 + 20) + (50 + 10 + 100 + 20 + 100 + 30), f"got {i}"

# the history is sent again every turn, so the input grows quadratically with the number of turns
i10, _ = chat_billed_tokens([50] * 10, [200] * 10)
i20, _ = chat_billed_tokens([50] * 20, [200] * 20)
assert 3.5 < i20 / i10 < 4.2, f"twice as many turns should cost about four times the input tokens, got {i20 / i10:.2f}"
assert i20 > 10 * sum([200] * 20), "after 20 turns the re-sent history far outweighs the generated text"

assert abs(cost(1_000_000, 0, 3.0, 15.0) - 3.0) < 1e-12
assert abs(cost(250_000, 20_000, 3.0, 15.0) - (0.75 + 0.30)) < 1e-12

assert fit_to_window([50, 100, 100, 100], 400) == [0, 1, 2, 3], "everything fits"
assert fit_to_window([50, 100, 100, 100], 300) == [0, 2, 3], "drop the oldest message after the system prompt"
assert fit_to_window([50, 100, 100, 100], 160) == [0, 3]
assert fit_to_window([50, 100, 100, 100], 200, keep_first=False) == [2, 3]
assert fit_to_window([50, 100, 100, 100], 10) == [0], "the system prompt stays even if nothing else fits"
print("m32_budget: all tests passed")
print(f"  10 turns: {i10:,} input tokens, 20 turns: {i20:,} input tokens")

<details><summary><b>提示 1 / Hint 1</b></summary>

用一个变量记录“到目前为止的历史长度”。每一轮先加上新的用户消息，把它计入累计输入，再加上这一轮的回答。

Keep one variable for the length of the history so far. Each turn first adds the new user message, counts the history towards the cumulative input and then adds the answer of the turn.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`fit_to_window` 维护一个保留下来的下标列表，只要总长度超限并且还有可以丢的消息，就删掉其中最早的一条。`keep_first` 为真时第 0 条不能删。

`fit_to_window` keeps a list of retained indices. While the total exceeds the limit and something may still be dropped, remove the oldest droppable message. With `keep_first` set, index 0 may not be removed.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import tiktoken


def count_tokens(enc, text):
    return len(enc.encode(text))


def chat_billed_tokens(user_lens, assistant_lens, system_len=0):
    total_in = total_out = 0
    history = system_len
    for u, a in zip(user_lens, assistant_lens):
        history += u                 # the new user message joins the history
        total_in += history          # the whole history is sent again
        total_out += a
        history += a                 # the answer becomes history for the next turn
    return total_in, total_out


def cost(input_tokens, output_tokens, price_in, price_out):
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def fit_to_window(message_lens, limit, keep_first=True):
    kept = list(range(len(message_lens)))
    start = 1 if keep_first else 0
    while sum(message_lens[i] for i in kept) > limit and len(kept) > start:
        kept.pop(start)              # the oldest message that may be dropped
    return kept
```

</details>

## 2. 信息的位置是否影响模型能否用上它 / Does the position of information affect whether the model uses it

给 GPT-2 一串 “The code for 某人 is 某数” 的事实，再问其中某一条。实现判断模型是否答对的函数，并统计正确率随这条事实所在位置的变化。测试会检查位于末尾的事实取回率远高于位于开头的。运行后请观察打印出的各位置正确率。

Give GPT-2 a list of facts of the form “The code for someone is some number” and ask about one of them. Implement the function that decides whether the model answers correctly and measure how accuracy changes with the position of the fact. The tests check that a fact at the end is retrieved far more often than one at the start. Look at the printed accuracy per position after running.

需要联网下载模型。/ Downloads a model.

In [ ]:
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NAMES = ["Alice", "Bob", "Carol", "David", "Emma", "Frank", "Grace", "Henry", "Irene", "Jack",
         "Karen", "Leo", "Maria", "Nick", "Olivia", "Paul", "Quinn", "Rachel", "Sam", "Tina"]


def build_prompt(names, codes, i):
    """A list of facts followed by a question about fact number i. Nothing to change."""
    context = "".join(f"The code for {n} is {c}.\n" for n, c in zip(names, codes))
    return context + f"The code for {names[i]} is"


@torch.no_grad()
def retrieval_correct(model, tok, names, codes, i):
    """Does the model pick the right code for names[i]?

    Build the prompt, take the logits of the last position, and compare ONLY the candidate
    tokens: the first token of ' ' + str(code) for every code in the list.
    Return True if the candidate with the largest logit belongs to codes[i].
    """
    # TODO 1
    raise NotImplementedError


def accuracy_by_position(model, tok, n_items=5, trials=12, seed=0):
    """Accuracy as a function of where the queried fact sits in the context.

    For trial t use  r = random.Random(seed + t),  names = r.sample(NAMES, n_items),
    codes = r.sample(range(10, 100), n_items)  (in this order), and query every position.
    Returns a list of n_items floats: the share of trials in which position i was answered correctly.
    """
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()

r = retrieval_correct(gpt2, tok, ["Alice", "Bob", "Carol"], [17, 42, 88], 2)
assert isinstance(r, bool)
assert r is True, "GPT-2 retrieves the most recent fact"

acc = accuracy_by_position(gpt2, tok, n_items=5, trials=12, seed=0)
assert len(acc) == 5 and all(0.0 <= a <= 1.0 for a in acc)
assert acc == accuracy_by_position(gpt2, tok, n_items=5, trials=12, seed=0), "same seed, same result"
assert acc[-1] > 0.7, f"the last fact should be retrieved most of the time, got {acc[-1]:.2f}"
assert acc[-1] > acc[0] + 0.4, f"the same kind of fact is retrieved far less often at the start ({acc[0]:.2f}) than at the end ({acc[-1]:.2f})"
print("m32_position: all tests passed")
print("  accuracy by position (first ... last):", [round(a, 2) for a in acc], " chance = 0.20")

<details><summary><b>提示 1 / Hint 1</b></summary>

候选 token 是每个数字前面加一个空格后分词得到的第一个 token，即 `tok(' ' + str(c)).input_ids[0]`。只在这些候选之间比较 logit。

The candidate tokens are the first token of every number with a space in front, `tok(' ' + str(c)).input_ids[0]`. Compare logits among these candidates only.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`logits[candidates].argmax() == i` 给出的是张量，用 `bool(...)` 转成 Python 的布尔值。

`logits[candidates].argmax() == i` is a tensor. Convert it to a Python boolean with `bool(...)`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

`accuracy_by_position` 对每次试验按给定方式生成名字和数字，再对每个位置调用一次 `retrieval_correct`，最后除以试验次数。

`accuracy_by_position` generates the names and numbers of each trial as specified, calls `retrieval_correct` once per position, and finally divides by the number of trials.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NAMES = ["Alice", "Bob", "Carol", "David", "Emma", "Frank", "Grace", "Henry", "Irene", "Jack",
         "Karen", "Leo", "Maria", "Nick", "Olivia", "Paul", "Quinn", "Rachel", "Sam", "Tina"]


def build_prompt(names, codes, i):
    context = "".join(f"The code for {n} is {c}.\n" for n, c in zip(names, codes))
    return context + f"The code for {names[i]} is"


@torch.no_grad()
def retrieval_correct(model, tok, names, codes, i):
    ids = tok(build_prompt(names, codes, i), return_tensors="pt").input_ids
    logits = model(ids).logits[0, -1]
    candidates = torch.tensor([tok(" " + str(c)).input_ids[0] for c in codes])
    return bool(logits[candidates].argmax() == i)


def accuracy_by_position(model, tok, n_items=5, trials=12, seed=0):
    hits = [0] * n_items
    for t in range(trials):
        r = random.Random(seed + t)
        names = r.sample(NAMES, n_items)
        codes = r.sample(range(10, 100), n_items)
        for i in range(n_items):
            hits[i] += retrieval_correct(model, tok, names, codes, i)
    return [h / trials for h in hits]
```

</details>

## 3. 估算一次长对话的 token 数 / Estimate the tokens of a long conversation

一个客服对话系统的 system prompt 有 800 个 token。每轮用户消息平均 60 个 token，回答平均 240 个 token。
(a) 第 20 轮的输入有多少个 token？
(b) 20 轮下来累计计费的输入 token 和输出 token 各是多少？
(c) 如果模型的上下文窗口是 8000 个 token，并且每轮要给回答留出 500 个 token，对话最多能进行到第几轮而不需要丢弃历史？
(d) 说出两种在超出窗口后处理历史的办法，以及各自的代价。

A customer-service chat system has a system prompt of 800 tokens. Each user message averages 60 tokens and each answer 240 tokens.
(a) How many tokens is the input of turn 20?
(b) Over 20 turns, how many input tokens and output tokens are billed in total?
(c) If the context window is 8000 tokens and 500 tokens are reserved for the answer of each turn, up to which turn can the conversation go without dropping history?
(d) Name two ways of handling the history once the window is exceeded, with the price of each.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 用 $\text{in}_i = s + i\,u + (i-1)\,a$。

(a) Use $\text{in}_i = s + i\,u + (i-1)\,a$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 解不等式 $s + i\,u + (i-1)\,a + 500 \le 8000$。

(c) Solve the inequality $s + i\,u + (i-1)\,a + 500 \le 8000$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $800 + 20 \times 60 + 19 \times 240 = 800 + 1200 + 4560 = 6560$。

(b) 输入为 $20 \times 800 + 60 \times \tfrac{20 \times 21}{2} + 240 \times \tfrac{20 \times 19}{2} = 16000 + 12600 + 45600 = 74200$。输出为 $20 \times 240 = 4800$。累计输入是累计输出的 15 倍以上。

(c) $800 + 60 i + 240 (i - 1) + 500 \le 8000$ 即 $300 i \le 6940$，$i \le 23.1$，所以最多到第 23 轮。

(d) 丢弃最早的消息，实现简单，代价是早期的信息完全丢失。把较早的对话压缩成摘要，能保留大意，代价是细节丢失，并且生成摘要本身需要一次额外的模型调用。

---

(a) $800 + 20 \times 60 + 19 \times 240 = 800 + 1200 + 4560 = 6560$.

(b) The input is $20 \times 800 + 60 \times \tfrac{20 \times 21}{2} + 240 \times \tfrac{20 \times 19}{2} = 16000 + 12600 + 45600 = 74200$. The output is $20 \times 240 = 4800$. The cumulative input is more than 15 times the cumulative output.

(c) $800 + 60 i + 240 (i - 1) + 500 \le 8000$ means $300 i \le 6940$, so $i \le 23.1$ and the conversation can reach turn 23.

(d) Dropping the oldest messages is simple to implement, at the price of losing early information completely. Compressing the earlier conversation into a summary keeps the gist, at the price of losing details and of one extra model call to produce the summary.

</details>